# Solutions — Java Streams → Comprehensions, Generators & Built-ins

Exercise notebook: `04_java_to_python/02_streams_vs_python.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · filter → map → collect

**Working solution**

In [ ]:
def big_spender_names(customers, threshold):
    return sorted(c["name"] for c in customers if c["spend"] > threshold)

**Explanation**

`sorted()` accepts any iterable, so a generator expression feeds it directly — no intermediate list.

**Complexity:** O(n log n).

**Common mistakes**

- `list(map(lambda c: c['name'], filter(lambda c: ..., customers)))` — legal, but nobody wants to read it.

## Exercise 2 — 🟡 Medium · groupingBy + summingDouble

**Working solution**

In [ ]:
from collections import defaultdict

def revenue_by_category(lines):
    totals = defaultdict(float)
    for line in lines:
        totals[line["category"]] += line["qty"] * line["price"]
    return {cat: round(total, 2) for cat, total in sorted(totals.items())}

**Explanation**

A collector is just an accumulator: a `defaultdict` plus a loop. Sorting at the end replaces `TreeMap::new`.

**Complexity:** O(n + k log k).

**Common mistakes**

- Trying to force this into a single comprehension with nested `sum()` per key — O(n·k).

## Exercise 3 — 🟡 Medium · partitioningBy

**Working solution**

In [ ]:
def partition(items, predicate):
    matching, rest = [], []
    for item in items:
        (matching if predicate(item) else rest).append(item)
    return matching, rest

**Explanation**

One pass, two lists, a tuple return that callers unpack: `valid, invalid = partition(rows, is_valid)`. Two comprehensions would call the predicate twice per item.

**Complexity:** O(n).

**Common mistakes**

- `[x for x in items if p(x)], [x for x in items if not p(x)]` — doubles the work and breaks for one-shot iterators.

## Exercise 4 — 🟠 Hard · flatMap + distinct + sorted + limit

**Working solution**

In [ ]:
from collections import Counter

def top_tags(orders, n=3):
    counts = Counter(tag for o in orders for tag in {t.lower() for t in (o.get("tags") or [])})
    return sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:n]

**Explanation**

`flatMap` = a second `for` clause in a generator expression. Per-order `distinct()` = a set comprehension. `Counter` consumes the
generator directly. Note: `Counter.most_common()` breaks ties by insertion order, not alphabetically — so sort explicitly.

**Complexity:** O(total tags + k log k).

**Common mistakes**

- Relying on `most_common` for deterministic ties.
- Counting duplicates within an order.

## Challenge — 🔴 Very Hard · Multiple collectors in one pass

**Working solution**

In [ ]:
def country_summary(orders):
    acc = {}
    for o in orders:
        s = acc.get(o["country"])
        if s is None:
            s = acc[o["country"]] = {"count": 0, "total": 0.0, "max_amount": float("-inf"), "max_order_id": None}
        s["count"] += 1
        s["total"] += o["amount"]
        if o["amount"] > s["max_amount"]:
            s["max_amount"], s["max_order_id"] = o["amount"], o["order_id"]
    return {country: {"count": s["count"], "total": round(s["total"], 2), "avg": round(s["total"] / s["count"], 2),
                      "max_order_id": s["max_order_id"]}
            for country, s in sorted(acc.items())}

**Explanation**

When you must aggregate several things in one pass, keep a small mutable state dict per key and finalize at the end — exactly what
Spark's `agg(count, sum, avg, max_by)` does per partition. Tuple assignment `a, b = x, y` updates two fields atomically.

**Complexity:** O(n) time, O(countries) memory.

**Common mistakes**

- Calling `list(orders)` first — defeats streaming on huge inputs.
- Computing avg from rounded totals.